# Week 5 Final: AI-Assisted Triage Data Exploration

I used the Week 5 triage CSV from TenX and treated "esi" as the triage level target. ESI 1 is the most urgent group and ESI 5 is the least urgent group, so I read the correlations with that direction in mind.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

repo_root = Path("..").resolve()
full_csv = repo_root / "week5" / "data" / "yaleemmlc_admissionprediction_triage.csv"
sample_csv = repo_root / "week5" / "data" / "yaleemmlc_triage_sample.csv"

csv_path = full_csv if full_csv.exists() else sample_csv
print(f"Using {csv_path.name}")

df = pd.read_csv(csv_path, low_memory=False)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df.shape


## Basic Profile

I checked the row count, column count, target distribution, and missingness before looking at plots. The reduced Week 5 file already has complete values across the included columns, so the bigger quality questions are class balance, plausible ranges, and whether the features make clinical sense for triage.


In [ ]:
esi = pd.to_numeric(df["esi"], errors="coerce")

profile = pd.DataFrame({
    "check": ["rows", "columns", "missing esi target", "duplicate rows"],
    "value": [len(df), df.shape[1], int(esi.isna().sum()), int(df.duplicated().sum())],
})
profile


In [ ]:
esi_counts = df["esi"].value_counts().sort_index()
esi_percent = (df["esi"].value_counts(normalize=True).sort_index() * 100).round(2)
pd.DataFrame({"visits": esi_counts, "percent": esi_percent})


In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False).reset_index()
missing.columns = ["feature", "missing_percent"]
missing.head(20)


## Clinical Range Checks

I did not delete these rows in the notebook. I flagged them because some values could be real emergencies while others could be unit or entry problems. A clinician should decide how to handle them before Week 6 modelling.


In [ ]:
range_checks = {
    "Heart rate outside 40-220 bpm": ((df["triage_vital_hr"] < 40) | (df["triage_vital_hr"] > 220)),
    "Systolic BP outside 70-250 mmHg": ((df["triage_vital_sbp"] < 70) | (df["triage_vital_sbp"] > 250)),
    "Diastolic BP outside 30-150 mmHg": ((df["triage_vital_dbp"] < 30) | (df["triage_vital_dbp"] > 150)),
    "Respiratory rate outside 8-40 breaths/min": ((df["triage_vital_rr"] < 8) | (df["triage_vital_rr"] > 40)),
    "Oxygen saturation outside 70-100 percent": ((df["triage_vital_o2"] < 70) | (df["triage_vital_o2"] > 100)),
    "Temperature outside 94-106 F": ((df["triage_vital_temp"] < 94) | (df["triage_vital_temp"] > 106)),
    "Glucose outside 40-500 mg/dL": ((df["triage_glucose"] < 40) | (df["triage_glucose"] > 500)),
}

range_summary = pd.DataFrame([
    {"check": name, "rows": int(mask.sum()), "percent": round(mask.mean() * 100, 3)}
    for name, mask in range_checks.items()
])
range_summary


## Chief Complaints and Feature Signals

The dataset stores chief complaints as separate yes/no flags. I checked the most common complaints, then compared selected features with ESI using simple correlations. This is not a model, but it helps decide whether the dataset has enough signal for Week 6.


In [ ]:
cc_cols = [col for col in df.columns if col.startswith("cc_")]
chief_complaints = df[cc_cols].sum().sort_values(ascending=False)
chief_complaints.head(20)


In [ ]:
candidate_cols = ["age", "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2", "triage_vital_temp", "triage_glucose"] + cc_cols

rows = []
for col in candidate_cols:
    series = pd.to_numeric(df[col], errors="coerce")
    corr = series.corr(esi)
    if pd.notna(corr):
        rows.append({"feature": col, "correlation_with_esi": corr, "absolute_correlation": abs(corr)})

feature_signal = pd.DataFrame(rows).sort_values("absolute_correlation", ascending=False)
feature_signal.head(20)


## Saved Outputs

I saved the final memo, plots, and tables in the repo so the submission can be reviewed without rerunning the notebook.

![Week 5 data quality dashboard](../docs/week5_data_quality_dashboard.svg)

![Week 5 feature signal summary](../docs/week5_feature_signal_summary.svg)
